# Two-Agent RAG — Document Q&A with Critic

**What this does:**
- Loads a local `.txt` document and chunks it into pieces
- `RAG_Answerer` retrieves relevant chunks and answers a question
- `Critic` checks the answer for accuracy and completeness
- No API key needed — runs fully on Ollama (llama3.2)

**Pre-requisite:** Ollama installed and `llama3.2` pulled.

In [1]:
%pip install -q autogen-agentchat autogen-ext[openai] openai

Note: you may need to restart the kernel to use updated packages.


In [1]:
import asyncio, os
from pathlib import Path
import json as _json, urllib.request
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

import autogen_agentchat
print('autogen-agentchat version:', autogen_agentchat.__version__)

autogen-agentchat version: 0.7.5


In [2]:
ollama_client = OpenAIChatCompletionClient(
    model='llama3.2',
    base_url='http://localhost:11434/v1',
    api_key='ollama',
    model_capabilities={'vision': False, 'function_calling': False, 'json_output': False},
)
print('Ollama client ready')

Ollama client ready


C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [3]:
try:
    with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
        tags = _json.loads(r.read())
        models = [m['name'] for m in tags.get('models', [])]
        print('Ollama running. Models:', models)
        if not any('llama3.2' in m for m in models):
            print('⚠️  Run in terminal: ollama pull llama3.2')
except Exception as e:
    print('❌ Ollama not reachable:', e)

Ollama running. Models: ['llama3.2:latest']


## Step 1 — Create a Sample Document

Replace content with your own text if needed.

In [4]:
doc_path = Path('rag_document.txt')

sample_text = '''
Artificial Intelligence (AI) refers to the simulation of human intelligence
in machines programmed to think and learn. AI systems are used in healthcare,
finance, transportation, and education.

Machine Learning (ML) is a subset of AI that allows systems to learn and
improve from experience without being explicitly programmed. Common ML
techniques include supervised learning, unsupervised learning, and reinforcement
learning.

Deep Learning is a subset of ML that uses neural networks with many layers
to model complex patterns in data. It powers image recognition, speech
recognition, and natural language processing.

Natural Language Processing (NLP) enables computers to understand, interpret,
and generate human language. NLP is used in chatbots, translation services,
sentiment analysis, and document summarization.

Retrieval-Augmented Generation (RAG) combines a retrieval system with a
language model. The retrieval system fetches relevant document chunks, which
are passed to the language model as context to generate accurate, grounded answers.
'''

doc_path.write_text(sample_text.strip())
print(f'Document written: {doc_path.resolve()}, {len(sample_text)} chars')

Document written: C:\Users\lhari\AI AGent\rag_document.txt, 1061 chars


## Step 2 — Chunk and Retrieve

In [5]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]


def simple_retrieve(query: str, chunks: list, top_k: int = 3) -> str:
    query_words = set(query.lower().split())
    scored = sorted(
        chunks,
        key=lambda c: len(query_words & set(c.lower().split())),
        reverse=True
    )
    return '\n\n---\n\n'.join(scored[:top_k])


document_text = doc_path.read_text()
chunks = chunk_text(document_text)
print(f'Split into {len(chunks)} chunks')

Split into 5 chunks


## Step 3 — Set Your Question

In [6]:
question = 'What is RAG and how does it work?'

retrieved_context = simple_retrieve(question, chunks, top_k=3)
print(f'Question: {question}')
print(f'\nRetrieved context:\n{"-"*40}\n{retrieved_context}')

Question: What is RAG and how does it work?

Retrieved context:
----------------------------------------
tems to learn and
improve from experience without being explicitly programmed. Common ML
techniques include supervised learning, unsupervised learning, and reinforcement
learning.

Deep Learning is a subset of ML that uses neural networks with many layers
to model complex patterns in data. It powers

---

ayers
to model complex patterns in data. It powers image recognition, speech
recognition, and natural language processing.

Natural Language Processing (NLP) enables computers to understand, interpret,
and generate human language. NLP is used in chatbots, translation services,
sentiment analysis, an

---

Artificial Intelligence (AI) refers to the simulation of human intelligence
in machines programmed to think and learn. AI systems are used in healthcare,
finance, transportation, and education.

Machine Learning (ML) is a subset of AI that allows systems to learn and
improve from 

## Step 4 — Create Two Agents

In [7]:
rag_answerer = AssistantAgent(
    name='RAG_Answerer',
    model_client=ollama_client,
    system_message=(
        'You are a precise Q&A assistant. Answer ONLY using the provided '
        'document context. If the context lacks information, say so clearly. '
        'Keep your answer concise and factual.'
    ),
)

critic = AssistantAgent(
    name='Critic',
    model_client=ollama_client,
    system_message=(
        'You are a critical reviewer. Check if the answer is accurate and '
        'grounded in the provided context. Point out any inaccuracies. '
        'If the answer is satisfactory, respond with: APPROVED. TERMINATE'
    ),
)
print('Agents created')

Agents created


## Step 5 — Run the Two-Agent RAG Chat

In [8]:
rag_task = (
    f'Question: {question}\n\n'
    f'Retrieved Context:\n{retrieved_context}\n\n'
    f'Please answer the question using only the context above.'
)

async def run_rag_chat():
    termination = TextMentionTermination('TERMINATE') | MaxMessageTermination(6)
    team = RoundRobinGroupChat([rag_answerer, critic], termination_condition=termination)
    result = await Console(team.run_stream(task=rag_task))
    await team.reset()
    return result

result = await run_rag_chat()

---------- TextMessage (user) ----------
Question: What is RAG and how does it work?

Retrieved Context:
tems to learn and
improve from experience without being explicitly programmed. Common ML
techniques include supervised learning, unsupervised learning, and reinforcement
learning.

Deep Learning is a subset of ML that uses neural networks with many layers
to model complex patterns in data. It powers

---

ayers
to model complex patterns in data. It powers image recognition, speech
recognition, and natural language processing.

Natural Language Processing (NLP) enables computers to understand, interpret,
and generate human language. NLP is used in chatbots, translation services,
sentiment analysis, an

---

Artificial Intelligence (AI) refers to the simulation of human intelligence
in machines programmed to think and learn. AI systems are used in healthcare,
finance, transportation, and education.

Machine Learning (ML) is a subset of AI that allows systems to learn and
improve from 

## Step 6 — Final Answer

In [9]:
print('\n=== FINAL ANSWER ===')
for msg in reversed(result.messages):
    if msg.source == 'RAG_Answerer':
        print(msg.content)
        break

print('\n=== CRITIC VERDICT ===')
for msg in reversed(result.messages):
    if msg.source == 'Critic':
        print(msg.content)
        break

print('\nStop reason:', result.stop_reason)


=== FINAL ANSWER ===
The context does not specifically explain what "RAG" stands for or how it works, within the provided information about Machine Learning (ML), Artificial Intelligence (AI) techniques. 

There is no mention of RAG in the given text, making it impossible to answer the question based on the provided context.

=== CRITIC VERDICT ===
APPROVED. TERMINATE

Stop reason: Text 'TERMINATE' mentioned
